<a href="https://colab.research.google.com/github/saadhana192465019/Digital-Forensics-and-cyber-crime-Investigation--CSA6102/blob/main/Experiment_48_AUTOMATED_GENERATION_OF_A_DIGITAL_FORENSIC_EXAMINATION_REPORT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================================================
# EXPERIMENT 10
# AUTOMATED GENERATION OF A DIGITAL FORENSIC EXAMINATION REPORT
# ================================================================

import hashlib
import os
import tempfile
from datetime import datetime, timezone, timedelta

# Indian Standard Time
IST = timezone(timedelta(hours=5, minutes=30))

# ------------------------------------------------
# MANDATORY REPORT SECTIONS
# ------------------------------------------------

MANDATORY_SECTIONS = [
    "1. CASE IDENTIFICATION",
    "2. AUTHORISATION AND SCOPE",
    "3. EXHIBITS RECEIVED",
    "4. TOOLS AND METHODS USED",
    "5. EVIDENCE INTEGRITY",
    "6. CHAIN OF CUSTODY SUMMARY",
    "7. FINDINGS OF FACT",
    "8. TIMELINE OF EVENTS",
    "9. OPINION",
    "10. LIMITATIONS",
    "11. DECLARATION OF THE EXAMINER"
]


# ------------------------------------------------
# CUSTOM ERROR
# ------------------------------------------------

class ReportError(Exception):
    pass


# ------------------------------------------------
# SHA-256 HASH FUNCTION
# ------------------------------------------------

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)

    return h.hexdigest()


# ------------------------------------------------
# BUILD FORENSIC REPORT
# ------------------------------------------------

def build_report(
    case,
    exhibits,
    tools,
    findings,
    timeline,
    opinion,
    limitations,
    examiner
):

    # Validate findings
    if not findings:
        raise ReportError(
            "A report cannot be issued with no findings of fact."
        )

    # Validate exhibits
    if not exhibits:
        raise ReportError(
            "A report cannot be issued with no exhibits."
        )

    # Validate exhibit hashes
    for exhibit in exhibits:

        if not exhibit.get("sha256"):
            raise ReportError(
                f"Exhibit {exhibit.get('id')} has no integrity hash."
            )

    # Report lines
    report = []

    A = report.append

    A("DIGITAL FORENSIC EXAMINATION REPORT")
    A("=" * 78)
    A("")

    # ------------------------------------------------
    # 1. CASE IDENTIFICATION
    # ------------------------------------------------

    A("1. CASE IDENTIFICATION")

    A(f"Case reference : {case['reference']}")
    A(f"Requesting authority : {case['authority']}")
    A(f"Date of report : {case['report_date']}")
    A("")

    # ------------------------------------------------
    # 2. AUTHORISATION AND SCOPE
    # ------------------------------------------------

    A("2. AUTHORISATION AND SCOPE")

    A(f"Authorisation : {case['authorisation']}")
    A("Questions referred for examination:")

    for question in case["questions"]:
        A(f"- {question}")

    A("")

    # ------------------------------------------------
    # 3. EXHIBITS RECEIVED
    # ------------------------------------------------

    A("3. EXHIBITS RECEIVED")

    for exhibit in exhibits:

        A(
            f"{exhibit['id']} : "
            f"{exhibit['description']}"
        )

        A(
            f"received {exhibit['received']} | "
            f"condition: {exhibit['condition']}"
        )

    A("")

    # ------------------------------------------------
    # 4. TOOLS AND METHODS USED
    # ------------------------------------------------

    A("4. TOOLS AND METHODS USED")

    for tool in tools:

        A(
            f"- {tool['name']} "
            f"v{tool['version']} : "
            f"{tool['purpose']}"
        )

    A("")

    # ------------------------------------------------
    # 5. EVIDENCE INTEGRITY
    # ------------------------------------------------

    A("5. EVIDENCE INTEGRITY")

    A(f"{'Exhibit':<10}{'SHA-256':<66}")

    for exhibit in exhibits:

        A(
            f"{exhibit['id']:<10}"
            f"{exhibit['sha256']}"
        )

    A(
        "Hashes were computed at acquisition and "
        "re-verified before analysis."
    )

    A("")

    # ------------------------------------------------
    # 6. CHAIN OF CUSTODY
    # ------------------------------------------------

    A("6. CHAIN OF CUSTODY SUMMARY")

    for custody in case["custody"]:
        A(custody)

    A("")

    # ------------------------------------------------
    # 7. FINDINGS OF FACT
    # ------------------------------------------------

    A("7. FINDINGS OF FACT")

    for i, finding in enumerate(findings, 1):

        A(
            f"{i}. {finding['statement']}"
        )

        A(
            f"Supported by: "
            f"{finding['evidence']}"
        )

    A("")

    # ------------------------------------------------
    # 8. TIMELINE
    # ------------------------------------------------

    A("8. TIMELINE OF EVENTS")

    for event in timeline:

        A(
            f"{event['time']} "
            f"{event['event']}"
        )

    A("")

    # ------------------------------------------------
    # 9. OPINION
    # ------------------------------------------------

    A("9. OPINION")

    for line in opinion:
        A(line)

    A("")

    # ------------------------------------------------
    # 10. LIMITATIONS
    # ------------------------------------------------

    A("10. LIMITATIONS")

    for limitation in limitations:
        A(f"- {limitation}")

    A("")

    # ------------------------------------------------
    # 11. EXAMINER DECLARATION
    # ------------------------------------------------

    A("11. DECLARATION OF THE EXAMINER")

    A(
        "I understand that my duty is to assist the court "
        "on matters within my expertise, and that this duty "
        "overrides any obligation to the party instructing me."
    )

    A(
        "The opinions expressed are my own and are based "
        "on the facts stated."
    )

    A(
        "Where I have relied on the work of others, "
        "that is stated."
    )

    A("")

    A(f"Name : {examiner['name']}")
    A(f"Qualifications : {examiner['qualifications']}")
    A(f"Designation : {examiner['designation']}")

    A(
        f"Signature : ____________________ "
        f"Date : {case['report_date']}"
    )

    A("=" * 78)

    return "\n".join(report)


# ------------------------------------------------
# REPORT VALIDATION
# ------------------------------------------------

def validate_report(text):

    missing_sections = []

    for section in MANDATORY_SECTIONS:

        if section not in text:
            missing_sections.append(section)

    return missing_sections


# ------------------------------------------------
# DEMO DATA
# ------------------------------------------------

def demo_inputs():

    # Create temporary evidence files
    temp_directory = tempfile.mkdtemp()

    evidence1 = os.path.join(
        temp_directory,
        "EX01.dd"
    )

    evidence2 = os.path.join(
        temp_directory,
        "EX02.bin"
    )

    # Create sample evidence
    with open(evidence1, "wb") as f:
        f.write(os.urandom(4096))

    with open(evidence2, "wb") as f:
        f.write(os.urandom(2048))

    # ------------------------------------------------
    # CASE INFORMATION
    # ------------------------------------------------

    case = {

        "reference": "CASE/CYB/2026/0417",

        "authority":
            "Inspector of Police, Cyber Crime Police Station, Chennai",

        "report_date":
            datetime.now(IST).strftime("%d-%m-%Y"),

        "authorisation":
            "Requisition dated 20-08-2026 under "
            "s.79A of the IT Act, 2000",

        "questions": [

            "Whether the exhibits contain evidence "
            "of unauthorised transfer of company data.",

            "Whether any audit records were deleted, "
            "and if so when."
        ],

        "custody": [

            "20-08-2026 09:00 IST seized by "
            "SI R. Menon at the branch office",

            "21-08-2026 10:30 IST imaged by "
            "Examiner A. Rao under write-block conditions"
        ]
    }

    # ------------------------------------------------
    # EXHIBITS
    # ------------------------------------------------

    exhibits = [

        {
            "id": "EX-01",

            "description":
                "Forensic image of laptop "
                "LT-9931-A (dd, 512 GB)",

            "received": "21-08-2026",

            "condition":
                "Sealed, seal intact",

            "sha256":
                sha256_file(evidence1)
        },

        {
            "id": "EX-02",

            "description":
                "Export of proxy logs, "
                "01-08-2026 to 20-08-2026",

            "received": "21-08-2026",

            "condition":
                "Received on write-once media",

            "sha256":
                sha256_file(evidence2)
        }
    ]

    # ------------------------------------------------
    # TOOLS
    # ------------------------------------------------

    tools = [

        {
            "name":
                "Write blocker (hardware)",

            "version":
                "T35u",

            "purpose":
                "Prevent modification of source media"
        },

        {
            "name":
                "dd / dcfldd",

            "version":
                "1.7",

            "purpose":
                "Bit-stream acquisition"
        },

        {
            "name":
                "Python hashlib",

            "version":
                "3.11",

            "purpose":
                "SHA-256 integrity verification"
        }
    ]

    # ------------------------------------------------
    # FINDINGS
    # ------------------------------------------------

    findings = [

        {
            "statement":
                "A file archive of 41,231,872 bytes was uploaded "
                "from host WS0142 to an external service at "
                "04:18 IST on 20-08-2026.",

            "evidence":
                "Proxy log entry, EX-02, line 4471"
        },

        {
            "statement":
                "A new account svc_helper was created on DC01 "
                "at 04:05 IST on 20-08-2026.",

            "evidence":
                "Windows Security Event ID 4720, EX-01"
        },

        {
            "statement":
                "The Security event log on FS01 was cleared "
                "at 04:31 IST on 20-08-2026.",

            "evidence":
                "Windows Security Event ID 1102, EX-01"
        }
    ]

    # ------------------------------------------------
    # TIMELINE
    # ------------------------------------------------

    timeline = [

        {
            "time":
                "20-08-2026 03:44 IST",

            "event":
                "First periodic outbound connection "
                "from 10.20.3.41"
        },

        {
            "time":
                "20-08-2026 04:05 IST",

            "event":
                "Account svc_helper created on DC01"
        },

        {
            "time":
                "20-08-2026 04:18 IST",

            "event":
                "41 MB archive uploaded "
                "to files.example.net"
        },

        {
            "time":
                "20-08-2026 04:31 IST",

            "event":
                "Security event log cleared on FS01"
        }
    ]

    # ------------------------------------------------
    # OPINION
    # ------------------------------------------------

    opinion = [

        "In my opinion the artefacts are consistent with "
        "an unauthorised transfer of data from the internal "
        "network, followed by an attempt to remove the "
        "audit record of it.",

        "I express no opinion on the identity of the person "
        "who operated the account."
    ]

    # ------------------------------------------------
    # LIMITATIONS
    # ------------------------------------------------

    limitations = [

        "Volatile memory was not available for EX-01, "
        "as the device was powered off on seizure.",

        "Proxy logs before 01-08-2026 were outside the "
        "retention period and could not be examined.",

        "Attribution to a natural person is beyond the "
        "scope of this examination."
    ]

    # ------------------------------------------------
    # EXAMINER
    # ------------------------------------------------

    examiner = {

        "name":
            "A. Rao",

        "qualifications":
            "M.Tech (Cyber Forensics), CHFI",

        "designation":
            "Scientific Officer, State Forensic Science Laboratory"
    }

    return (
        case,
        exhibits,
        tools,
        findings,
        timeline,
        opinion,
        limitations,
        examiner
    )


# ================================================================
# TEST CASES
# ================================================================

def run_tests():

    args = demo_inputs()

    (
        case,
        exhibits,
        tools,
        findings,
        timeline,
        opinion,
        limitations,
        examiner
    ) = args

    # Generate report
    report = build_report(*args)

    results = []

    # ------------------------------------------------
    # TC1
    # ------------------------------------------------

    results.append(
        (
            "TC1 all 11 mandatory sections present",
            validate_report(report) == []
        )
    )

    # ------------------------------------------------
    # TC2
    # ------------------------------------------------

    results.append(
        (
            "TC2 every exhibit hash printed",
            all(
                e["sha256"] in report
                for e in exhibits
            )
        )
    )

    # ------------------------------------------------
    # TC3
    # ------------------------------------------------

    results.append(
        (
            "TC3 every finding printed",
            all(
                f["statement"][:40] in report
                for f in findings
            )
        )
    )

    # ------------------------------------------------
    # TC4
    # ------------------------------------------------

    results.append(
        (
            "TC4 limitations section is not empty",
            all(
                l[:30] in report
                for l in limitations
            )
        )
    )

    # ------------------------------------------------
    # TC5
    # ------------------------------------------------

    results.append(
        (
            "TC5 expert duty-to-court declaration present",

            "duty is to assist the court" in report
            and
            "overrides any obligation to the party"
            in report
        )
    )

    # ------------------------------------------------
    # TC6
    # ------------------------------------------------

    results.append(
        (
            "TC6 opinion separated from fact",

            report.index("7. FINDINGS OF FACT")
            <
            report.index("9. OPINION")
        )
    )

    # ------------------------------------------------
    # TC7 - Empty findings
    # ------------------------------------------------

    try:

        build_report(
            case,
            exhibits,
            tools,
            [],
            timeline,
            opinion,
            limitations,
            examiner
        )

        raised = False

    except ReportError:

        raised = True

    results.append(
        (
            "TC7 empty findings rejected",
            raised
        )
    )

    # ------------------------------------------------
    # TC8 - Missing hash
    # ------------------------------------------------

    bad_exhibits = [
        dict(exhibits[0])
    ]

    bad_exhibits[0]["sha256"] = ""

    try:

        build_report(
            case,
            bad_exhibits,
            tools,
            findings,
            timeline,
            opinion,
            limitations,
            examiner
        )

        raised2 = False

    except ReportError:

        raised2 = True

    results.append(
        (
            "TC8 exhibit without hash rejected",
            raised2
        )
    )

    # ------------------------------------------------
    # TC9 - Missing section
    # ------------------------------------------------

    truncated = report.replace(
        "10. LIMITATIONS",
        "10. XXXX"
    )

    results.append(
        (
            "TC9 validator detects missing section",

            "10. LIMITATIONS"
            in validate_report(truncated)
        )
    )

    # ------------------------------------------------
    # TC10 - Report length
    # ------------------------------------------------

    results.append(
        (
            "TC10 report length reasonable",
            len(report.split("\n")) > 45
        )
    )

    # ------------------------------------------------
    # PRINT RESULTS
    # ------------------------------------------------

    print("\n" + "=" * 78)
    print("TEST CASE RESULTS")
    print("=" * 78)

    for name, passed in results:

        status = "PASS" if passed else "FAIL"

        print(
            f"{name:<50} -> {status}"
        )

    passed_count = sum(
        1 for _, ok in results if ok
    )

    print("=" * 78)

    print(
        f"RESULT: {passed_count}/{len(results)} "
        "test cases passed"
    )

    return report


# ================================================================
# RUN EXPERIMENT
# ================================================================

print("=" * 78)
print("EXPERIMENT 10")
print("AUTOMATED GENERATION OF A DIGITAL FORENSIC EXAMINATION REPORT")
print("=" * 78)

report = run_tests()


# ================================================================
# DISPLAY GENERATED REPORT
# ================================================================

print("\n\n")
print("=" * 78)
print("GENERATED FORENSIC REPORT")
print("=" * 78)

print(report)


# ================================================================
# VALIDATE REPORT
# ================================================================

missing = validate_report(report)

print("\n")
print("=" * 78)
print("REPORT VALIDATION")
print("=" * 78)

if not missing:

    print("VALIDATION SUCCESSFUL")
    print("All 11 mandatory sections are present.")

else:

    print("VALIDATION FAILED")
    print("Missing sections:")

    for section in missing:
        print("-", section)


# ================================================================
# SAVE REPORT
# ================================================================

report_file = "forensic_report.txt"

with open(report_file, "w", encoding="utf-8") as f:
    f.write(report)

print("\nReport saved as:", report_file)


# ================================================================
# DOWNLOAD REPORT IN COLAB
# ================================================================

try:

    from google.colab import files

    files.download(report_file)

except:

    print(
        "Download is available automatically when running "
        "this notebook in Google Colab."
    )

EXPERIMENT 10
AUTOMATED GENERATION OF A DIGITAL FORENSIC EXAMINATION REPORT

TEST CASE RESULTS
TC1 all 11 mandatory sections present              -> PASS
TC2 every exhibit hash printed                     -> PASS
TC3 every finding printed                          -> PASS
TC4 limitations section is not empty               -> PASS
TC5 expert duty-to-court declaration present       -> PASS
TC6 opinion separated from fact                    -> PASS
TC7 empty findings rejected                        -> PASS
TC8 exhibit without hash rejected                  -> PASS
TC9 validator detects missing section              -> PASS
TC10 report length reasonable                      -> PASS
RESULT: 10/10 test cases passed



GENERATED FORENSIC REPORT
DIGITAL FORENSIC EXAMINATION REPORT

1. CASE IDENTIFICATION
Case reference : CASE/CYB/2026/0417
Requesting authority : Inspector of Police, Cyber Crime Police Station, Chennai
Date of report : 20-08-2026

2. AUTHORISATION AND SCOPE
Authorisation : Requis

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>